# Stage 4, before the GPU: one frame per video, from one pipeline

**The question.** Stage 4 asks whether a video's first frame predicts its lift. Before any image model runs, this
notebook makes sure the frames can't give the target away by *how* they were made.

The trap (README trap 3): the platform keeps each channel's top clips by share rate and retires the rest. A retired
clip keeps only a poster, made by the platform's own code before it deleted the file: the first frame decoded after
seeking 1 s in (so the keyframe at or before 1 s), shrunk to 480 px on its longest edge with Lanczos, saved as a JPEG
at quality 82. A kept clip still has its whole video. If kept clips' frames were taken any other way (the frame at
exactly 1 s, another size, another JPEG quality), an image model could learn "this frame was made the other way",
which means "this clip was kept", which means "this clip was shared more". It would score by spotting the pipeline,
not the content.

So `sharerate.frames` makes kept clips' frames with the poster code itself, copied line for line, under the library
versions every poster was made with (PyAV 17.0.1, Pillow 12.1.1), and gives retired clips their posters, byte for
byte. Every frame is named by a random id; the map from id to video stays in `data/frames.sqlite`.

**What must hold**, written before the full check ran (a spot check of 60 clips had shown 59 frames byte-identical to
their posters, the 60th's poster older than its file):

1. **The poster match.** Where a clip has both a video file and a poster, and the poster was made from that same file
   (the poster's file date is not older than the video's), the frame made here is byte for byte the poster. If it
   isn't, the pipeline is fixed until it is.
2. **The frame-or-poster classifier.** On the clips that have both, a classifier given my frame and the platform's
   poster of the same clip can't tell them apart: the 95% creator-bootstrap interval of its out-of-fold ROC-AUC
   includes 0.5. The same videos on both sides, so only the pipeline differs.
3. **The positive control.** The same classifier, given the poster and a frame made the obvious way (the frame at
   exactly 1 s, resized with bicubic, JPEG quality 95), tells them apart: its interval lies well above 0.5. Without
   it, a 0.5 above could just mean a classifier that sees nothing.

## Setup

What it does: loads every target video's split and creator group, where its video file and poster are, and the
frame `python -m sharerate.frames` made for it. Why: every section below counts or compares these. The mistake it
avoids: the paths name the creator (the flywheel stores files by username), so nothing here ever prints a path, an
id or a row; only counts. No label is read except the training and validation lift in section 1.

In [1]:
import io
import os
from concurrent.futures import ProcessPoolExecutor
from contextlib import closing

import numpy as np
import pandas as pd
from PIL import Image, JpegImagePlugin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sharerate import data, frames, metrics
from sharerate.bootstrap import paired_creator_bootstrap

SEED, N_BOOT = 2026, 2_000  # the seed and number of creator draws of stages 2 and 3
SPLITS = ["train", "validation", "test"]
GROUPS = ["seen", "new", "holdout"]  # targets.creator_seen; training rows are all seen or holdout
MIN_CREATORS = 5  # rule 9: no number is shown for fewer than 5 creators
FW = frames.FLYWHEEL_DIR  # read-only: files are opened for reading, never written
pd.set_option("display.width", 200)
frames.check_versions()  # PyAV 17.0.1 and Pillow 12.1.1, or stop

files = frames.target_files()  # all three splits; split and group columns, no label
with closing(frames._open_map()) as con:
    made = pd.read_sql("SELECT username, video_id, frame_id, source, error FROM frames", con)
with closing(data._open_snapshot()) as con:
    codes = data._creator_codes(con)  # the opaque creator numbers every notebook uses
table = files.merge(made, on=["username", "video_id"], how="left", validate="one_to_one")
table["creator"] = table.username.map(codes)
table = table.drop(columns=["username", "video_id"])  # from here on, no name and no video id, only the paths
table["has_frame"] = table.source.notna()
table["from_video"] = table.source.eq("video")

print(f"{len(table):,} target videos from {table.creator.nunique()} creators")
print(table.source.value_counts(dropna=False).rename("frames by where they came from").to_string())
print(f"{table.error.notna().sum():,} videos logged an error while their frame was made")

143,489 target videos from 231 creators
source
poster    97899
video     45542
NaN          48
48 videos logged an error while their frame was made


## 1. Coverage

What it does: counts, for each split and creator group, how many target videos got a frame, and how many of those
frames were made from a video file rather than copied from a poster. Why: rule 5 says models compared in one table are
scored on the same rows, so if some videos have no frame, every earlier model must be re-scored on the rest. The
mistake it avoids: counting from the snapshot's `files` flags instead of from the frames actually made; a video
whose file is on disk but won't decode would be counted and have no frame.

In [2]:
coverage = table.groupby(["time_split", "creator_seen"]).agg(
    creators=("creator", "nunique"), videos=("has_frame", "size"), with_frame=("has_frame", "sum"),
    from_video=("from_video", "sum"))
coverage = coverage.reindex([(s, g) for s in SPLITS for g in GROUPS if (s, g) in coverage.index])
total = coverage.sum().to_frame("all").T
total["creators"] = table.creator.nunique()
coverage = pd.concat([coverage, total.set_index(pd.MultiIndex.from_tuples([("all", "")]))])
coverage["with_frame_%"] = (100 * coverage.with_frame / coverage.videos).round(2)
coverage["from_video_%"] = (100 * coverage.from_video / coverage.with_frame).round(1)
coverage

creators  videos  with_frame  from_video  with_frame_%  from_video_%
train      seen          148   74552       74518       20316         99.95          27.3
           holdout        35   25865       25856        7601         99.97          29.4
validation seen          138   13646       13645        4718         99.99          34.6
           new            27    2275        2274        1294         99.96          56.9
           holdout        38    5411        5409        2259         99.96          41.8
test       seen          138   12455       12454        4389         99.99          35.2
           new            39    4901        4901        3054        100.00          62.3
           holdout        42    4384        4384        1911        100.00          43.6
all                      231  143489      143441       45542         99.97          31.7

What it does: says why some videos have no frame, and compares their lift with the rest's on the training and
validation rows, with a 95% interval from 2,000 creator draws. Why: a model scored only on videos with a frame is
scored on a different population, and rule 5 needs to know how it differs. The mistake it avoids: reading the test's
lift to answer it; the test stays closed until stage 5, so this looks at training and validation only.

In [3]:
none = table[~table.has_frame]
reasons = none.error.fillna("").str.split(": ", n=2).str[-1].value_counts()
print(f"{len(none):,} target videos have no frame, from {none.creator.nunique()} creators; why:")
print(reasons.to_string())

videos = data.load_videos()  # training and validation only
paths = frames.frame_paths()  # the same rows, in the same order
without = paths.isna().to_numpy()
lift = videos.lift.to_numpy()
codes, _ = pd.factorize(videos.creator)
n_creators = codes.max() + 1
print(f"Training and validation: {without.sum():,} of {len(videos):,} videos have no frame, "
      f"from {videos.creator[without].nunique()} creators")
if videos.creator[without].nunique() >= MIN_CREATORS:
    # Per-creator sums, so each draw of creators is one matrix product: draws x creators times creators.
    sums = {name: (np.bincount(codes, lift * mask, n_creators), np.bincount(codes, mask * 1.0, n_creators))
            for name, mask in (("without", without), ("with", ~without))}
    rng = np.random.default_rng(SEED)
    draws = np.stack([np.bincount(rng.integers(0, n_creators, n_creators), minlength=n_creators)
                      for _ in range(N_BOOT)])
    mean = {name: (draws @ total) / (draws @ count) for name, (total, count) in sums.items()}
    gaps = mean["without"] - mean["with"]  # a draw with no frameless video gives nan, and is left out
    gap = lift[without].mean() - lift[~without].mean()
    low, high = np.nanquantile(gaps, [0.025, 0.975])
    print(f"Their mean lift minus the rest's: {gap:+.3f} log10 [{low:+.3f}, {high:+.3f}]")

48 target videos have no frame, from 29 creators; why:
error
File has no video stream.    48


Training and validation: 47 of 121,749 videos have no frame, from 28 creators
Their mean lift minus the rest's: -0.125 log10 [-0.284, -0.001]


## 2. The poster match

What it does: for every clip that has both a video file and a poster on disk, compares the frame made here (from
the video) with the platform's poster: whether the bytes are identical, and where they aren't, the image size, the
JPEG quantisation tables (the quality setting, as stored in the file), the chroma subsampling and the pixel difference.
It splits the clips by whether the poster was made from the file now on disk ("fresh": the poster's file date is not
older than the video's, the platform's own freshness test) or from an earlier download of the same clip ("stale").

Why: identical bytes settle it outright, so the finer measures only matter where bytes differ. The mistake it
avoids: blaming the pipeline for a stale poster. A stale poster was made from another copy of the video (a later
re-download replaced the file), so its frame can differ for reasons that have nothing to do with how frames are made.

In [4]:
def on_disk(relative):
    return isinstance(relative, str) and (FW / relative).is_file()


both = table[table.video_path.map(on_disk) & table.poster_path.map(on_disk)].copy()
assert both.from_video.all()  # a clip with a file got its frame from the file
both["mine"] = both.frame_id.map(lambda i: (frames.FRAMES_DIR / f"{i}.jpg").read_bytes())
both["theirs"] = both.poster_path.map(lambda p: (FW / p).read_bytes())
both["fresh"] = [os.stat(FW / p).st_mtime >= os.stat(FW / v).st_mtime
                 for p, v in zip(both.poster_path, both.video_path)]


def compare(mine, theirs):
    # Bytes first; the rest describes how two different files differ.
    a, b = Image.open(io.BytesIO(mine)), Image.open(io.BytesIO(theirs))
    row = {"identical": mine == theirs, "same_size": a.size == b.size,
           "same_tables": a.quantization == b.quantization,
           "same_subsampling": JpegImagePlugin.get_sampling(a) == JpegImagePlugin.get_sampling(b)}
    if row["same_size"]:
        diff = np.abs(np.asarray(a.convert("RGB"), np.int16) - np.asarray(b.convert("RGB"), np.int16))
        row.update(mean_abs_diff=diff.mean(), share_of_pixels_differing=(diff.max(axis=2) > 0).mean())
    return row


match = pd.DataFrame([compare(m, t) for m, t in zip(both.mine, both.theirs)], index=both.index)
both = both.join(match)
def describe_match(rows):
    return {"clips": len(rows), "creators": rows.creator.nunique(), "byte_identical": rows.identical.sum(),
            "same_size": rows.same_size.sum(), "same_tables": rows.same_tables.sum(),
            "same_subsampling": rows.same_subsampling.sum()}


groups = {"all clips with both": both, "fresh poster": both[both.fresh], "stale poster": both[~both.fresh]}
summary = pd.DataFrame({name: describe_match(rows) for name, rows in groups.items()
                        if rows.creator.nunique() >= MIN_CREATORS}).T
summary["byte_identical_%"] = (100 * summary.byte_identical / summary.clips).astype(float).round(2)
stale = both[~both.fresh]
if stale.creator.nunique() < MIN_CREATORS:
    print(f"{len(stale):,} clips have a poster older than their file. They come from fewer than {MIN_CREATORS} "
          "creators, so they are counted in the first row and not shown apart (rule 9).")
summary

290 clips have a poster older than their file. They come from fewer than 5 creators, so they are counted in the first row and not shown apart (rule 9).


,clips,creators,byte_identical,same_size,same_tables,same_subsampling,byte_identical_%
all clips with both,3882,151,3849,3882,3882,3882,99.15
fresh poster,3592,150,3592,3592,3592,3592,100.00


What it does: describes the clips whose frame and poster are not identical: how far apart their pixels are. Why: a
mean difference of a grey level or two on a 0-255 scale is two encodes of slightly different pictures; a different
pipeline shows up in the tables or the size first. The mistake it avoids: reading the pixel difference of all clips,
which the identical ones would pull to 0.

In [5]:
differ = both[~both.identical]
print(f"{len(differ):,} clips differ: {(~differ.fresh).sum():,} with a stale poster, "
      f"{differ.fresh.sum():,} with a fresh one")
for name, rows in (("stale", differ[~differ.fresh]), ("fresh", differ[differ.fresh])):
    if not len(rows):
        continue
    if rows.creator.nunique() < MIN_CREATORS:
        print(f"The {name} ones come from fewer than {MIN_CREATORS} creators: not described (rule 9)")
        continue
    display(rows[["same_size", "same_tables", "same_subsampling", "mean_abs_diff",
                  "share_of_pixels_differing"]].astype(float).describe(percentiles=[0.5]).round(3))

33 clips differ: 33 with a stale poster, 0 with a fresh one
The stale ones come from fewer than 5 creators: not described (rule 9)


## 3. The frame-or-poster classifier

The match above compares files. A model sees pictures, and could pick up a difference no single measure names, so
the check that settles it is a classifier: given an image, was it made here, or by the platform? It sees, for each
clip that has both, two rows: my frame (label 1) and the platform's poster (label 0).

Three kinds of evidence, each its own feature set:

- **Forensic**: what a different pipeline changes first. The image size, the file's size and bits per pixel, the
  quantisation tables' sums, the subsampling, brightness and contrast, sharpness (the variance of a Laplacian
  filter) and blockiness (how much the picture jumps across JPEG's 8-pixel block edges, against inside them).
- **CLIP**: what a frozen image encoder sees, the kind stage 4 will use: LAION's CLIP ViT-B/32, 512 numbers.
- **Both together.**

What this cell does: computes the forensic features of an image. Why these: compression and resizing leave their
traces exactly here. The mistake it avoids: features that read the file's name or date, which would detect
the folder rather than the picture.

In [6]:
def forensic(jpeg):
    image = Image.open(io.BytesIO(jpeg))
    tables = image.quantization
    rgb = np.asarray(image.convert("RGB"), dtype=np.float32)
    grey = rgb.mean(axis=2)
    laplacian = 4 * grey[1:-1, 1:-1] - grey[:-2, 1:-1] - grey[2:, 1:-1] - grey[1:-1, :-2] - grey[1:-1, 2:]
    step = np.abs(np.diff(grey, axis=1))  # each pixel against its right-hand neighbour
    across = step[:, 7::8].mean()  # between columns 8k+7 and 8k+8: across a JPEG block edge
    inside = np.delete(step, np.s_[7::8], axis=1).mean()
    return {"width": image.width, "height": image.height, "bytes": len(jpeg),
            "bits_per_pixel": 8 * len(jpeg) / (image.width * image.height),
            "luma_table": sum(tables[0]), "chroma_table": sum(tables.get(1, [0])),
            "subsampling": JpegImagePlugin.get_sampling(image),
            "red": rgb[..., 0].mean(), "green": rgb[..., 1].mean(), "blue": rgb[..., 2].mean(),
            "contrast": grey.std(), "sharpness": laplacian.var(), "blockiness": across / (inside + 1e-6)}


features = {"mine": pd.DataFrame([forensic(j) for j in both.mine]),
            "theirs": pd.DataFrame([forensic(j) for j in both.theirs])}
print(f"{features['mine'].shape[1]} forensic features for {len(both):,} clips, two images each")

13 forensic features for 3,882 clips, two images each


What it does: embeds every distinct image with CLIP ViT-B/32 on the CPU. Why distinct: a frame identical to its
poster is the same bytes, so it gets the same vector; embedding it once halves the work. The mistake it avoids:
embedding a frame and its identical poster separately and reading a floating-point wobble as a difference.

In [7]:
import warnings

import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore", message="IProgress not found")  # Jupyter's widgets, not the data
hf_logging.disable_progress_bar()  # no loading bar in the saved outputs

CLIP = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"
torch.set_num_threads(os.cpu_count())
clip = AutoModel.from_pretrained(CLIP).eval()
processor = AutoImageProcessor.from_pretrained(CLIP)


def clip_vectors(jpegs, batch=64):
    # One vector per distinct byte string, then looked up for every image.
    distinct = list(dict.fromkeys(jpegs))
    found = {}
    with torch.inference_mode():
        for start in range(0, len(distinct), batch):
            chunk = distinct[start:start + batch]
            images = [Image.open(io.BytesIO(j)).convert("RGB") for j in chunk]
            out = clip.get_image_features(**processor(images=images, return_tensors="pt")).pooler_output
            found.update(zip(chunk, out.numpy()))
    return np.stack([found[j] for j in jpegs])


vectors = {"mine": clip_vectors(list(both.mine)), "theirs": clip_vectors(list(both.theirs))}
print(f"{len(set(both.mine) | set(both.theirs)):,} distinct images embedded, {vectors['mine'].shape[1]} numbers each")

3,913 distinct images embedded, 512 numbers each


What it does: makes the positive control's frames, the obvious way (`frames.naive_frame`: the frame at exactly
1 s, bicubic, JPEG quality 95), from the same video files, in parallel, and gives them the same features. Why: a
check is only worth something if it can fail; this is a pipeline a careless stage 4 might have used. The mistake it
avoids: trusting a classifier's 0.5 without showing it can score far above 0.5 when there is something to find.

In [8]:
video_files = [FW / v for v in both.video_path]
with ProcessPoolExecutor(max_workers=6) as pool:
    naive = list(pool.map(frames.naive_frame, video_files, chunksize=16))
features["naive"] = pd.DataFrame([forensic(j) for j in naive])
vectors["naive"] = clip_vectors(naive)
print(f"{len(naive):,} naive frames; byte-identical to their poster: {sum(n == t for n, t in zip(naive, both.theirs))}")

3,882 naive frames; byte-identical to their poster: 0


What it does: fits each classifier out of fold and scores it. Folds are grouped by creator (`GroupKFold`, 5 folds),
so a clip's two images, and all of a creator's clips, are always predicted by a model that never saw that creator.
Each ROC-AUC carries the 95% interval of 2,000 creator draws (rule 6), through the same paired bootstrap as stages 2
and 3, against a constant score (whose ROC-AUC is 0.5).

Why logistic regression and gradient boosting: the first finds any consistent direction in many features, such as
CLIP's 512; the second finds thresholds in a few, such as "bits per pixel above x". The mistake it avoids: plain
k-fold, which would let the model learn one creator's look from their other clips and call it a pipeline.

In [9]:
def score(label, made_here, poster, creator_of_clip):
    # made_here and poster: (forensic features, CLIP vectors) of the same clips, in the same order.
    n = len(creator_of_clip)
    groups = np.r_[creator_of_clip, creator_of_clip]  # a clip's two rows share its creator, so its fold
    y = np.r_[np.ones(n), np.zeros(n)]  # 1: made here, 0: the platform's poster
    forensic_X = np.vstack([made_here[0], poster[0]])
    clip_X = np.vstack([made_here[1], poster[1]])
    sets = {"forensic, logistic": forensic_X, "forensic, boosting": forensic_X, "CLIP, logistic": clip_X,
            "both, logistic": np.hstack([forensic_X, clip_X])}
    rows = []
    for name, X in sets.items():
        model = (HistGradientBoostingClassifier(random_state=SEED) if "boosting" in name else
                 make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5_000)))
        oof = cross_val_predict(model, X, y, groups=groups, cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
        # metrics.roc_auc counts a row as positive when its first argument is at least log10(2); 1 and 0 do that.
        result = paired_creator_bootstrap(metrics.roc_auc, y, {"constant": np.zeros_like(y), "classifier": oof},
                                          groups, base="constant", n_boot=N_BOOT, seed=SEED).loc["classifier"]
        ranked_right = np.mean(np.sign(oof[:n] - oof[n:]) / 2 + 0.5)  # my image above its poster; a tie is half
        rows.append({"comparison": label, "features": name, "roc_auc": result.value, "low": result.value_low,
                     "high": result.value_high, "pairs_ranked_right": ranked_right})
    return rows


def evidence(kind, keep=slice(None)):
    return features[kind].to_numpy()[keep], vectors[kind][keep]


creators = both.creator.to_numpy()
results = pd.DataFrame(score("my frame vs its poster", evidence("mine"), evidence("theirs"), creators)
                       + score("control: naive vs poster", evidence("naive"), evidence("theirs"), creators))
results.round(3)

,comparison,features,roc_auc,low,high,pairs_ranked_right
0,my frame vs its poster,"forensic, logistic",0.5,0.5,0.5,0.500
1,my frame vs its poster,"forensic, boosting",0.5,0.5,0.5,0.500
2,my frame vs its poster,"CLIP, logistic",0.5,0.5,0.5,0.500
3,my frame vs its poster,"both, logistic",0.5,0.5,0.5,0.500
4,control: naive vs poster,"forensic, logistic",1.0,1.0,1.0,1.000
5,control: naive vs poster,"forensic, boosting",1.0,1.0,1.0,1.000
6,control: naive vs poster,"CLIP, logistic",1.0,1.0,1.0,0.999
7,control: naive vs poster,"both, logistic",1.0,1.0,1.0,1.000


What it does: the same classifiers on the fresh-poster clips alone, where the poster was made from the file the
frame was made from, so the pipeline is the only thing that can differ. Why: the stale posters come from another copy
of the video, and the table above mixes them in. The mistake it avoids: a headline that mixes two causes. The stale
clips can't be scored alone: they come from fewer than 5 creators, too few for folds by creator, and rule 9 forbids
showing them.

In [10]:
fresh = both.fresh.to_numpy()
print(f"Fresh posters: {fresh.sum():,} clips from {both.creator[fresh].nunique()} creators")
by_poster = pd.DataFrame(score("fresh posters only", evidence("mine", fresh), evidence("theirs", fresh),
                               creators[fresh]))
by_poster.round(3)

Fresh posters: 3,592 clips from 150 creators


,comparison,features,roc_auc,low,high,pairs_ranked_right
0,fresh posters only,"forensic, logistic",0.5,0.5,0.5,0.5
1,fresh posters only,"forensic, boosting",0.5,0.5,0.5,0.5
2,fresh posters only,"CLIP, logistic",0.5,0.5,0.5,0.5
3,fresh posters only,"both, logistic",0.5,0.5,0.5,0.5


## 4. What it shows

- **Every target video but 48 has a frame:** 143,441 of 143,489 (99.97%), 45,542 made from the video and 97,899
  posters (cell 2). The 48 are files with no video stream, so the platform could never have made a poster of them
  either. In training and validation they are 47 of 121,749 videos, from 28 creators, and their lift is lower than the
  rest's, −0.125 log10 [−0.284, −0.001] (cell 6): not a random 47, but too few to move a model's score. Stage 4 scores
  on the rows with a frame and re-scores the reference on the same rows (rule 5).
- **Most frames are posters, and newer videos still have their file more often:** 27% to 29% of the training period's
  frames come from a video, 35% to 57% of validation's and 35% to 62% of the test's (cell 4). So where a frame came from
  follows time as well as the target, one more reason it never becomes a feature (`frames.frame_paths` doesn't return it).
- **The poster match: every fresh poster is byte for byte the frame made here,** 3,592 of 3,592 clips from 150
  creators (cell 8). Over all 3,882 clips that have both, 3,849 (99.15%) are identical, and all 3,882 have the same
  size, quantisation tables and chroma subsampling. The 33 that differ all have a stale poster, made from an earlier
  download of the clip that a later download replaced (cell 10); they come from fewer than 5 creators, so their pixel
  differences aren't shown (rule 9). The pipeline needed no fixing.
- **The classifier can't tell my frames from the platform's posters:** ROC-AUC 0.500 [0.500, 0.500] with every
  feature set, forensic, CLIP or both (cell 18), and on the fresh posters alone (cell 20). Identical bytes force that:
  an image and its exact copy get the same score.
- **It is not a blind classifier.** Given the poster and a frame made the obvious way (the frame at exactly 1 s,
  bicubic, JPEG quality 95), the same classifiers score 1.000 [1.000, 1.000] to three decimals, **CLIP alone
  included**, which ranks 99.9% of the pairs right (cell 18).
- **What the control means for stage 4.** A frozen CLIP, the kind of encoder stage 4 will use, tells a frame taken
  the obvious way from the platform's poster almost every time. Had kept clips' frames been taken that way, an image
  model could have found the kept clips, and through them the target, without looking at what the picture shows.
  One pipeline closes that door.
- **What it doesn't settle.** The frames are made the same way, but kept and retired clips still differ in what they
  show: kept clips are the ones shared more, and that is the signal stage 4 looks for. Stage 4 still runs stage 3's
  leak check on the pixels: given a clip's lift, the image model's correction must not know whether the clip was kept.

The test's frames were counted (cell 4); the test's lift was never read in this notebook.